In [ ]:
# 准备训练、评测与播放录像的工具。
import lightning as L
import torch
from IPython.display import Video

from training.evaluate import evaluate, record_video
# 沿用仿真与训练骨架，PPO 也复用上一讲的优势估计方法。
from training.ppo import ActorCritic, PolicyModule, RolloutData, compute_gae, flat, normalize
from training.sim import SKILLS, DuckEnv, hardware

In [ ]:
# 保持选修直走任务和并行数不变，只比较算法。
hw = hardware.load("2026-11-26-hd1910")
skill = SKILLS["course-walk-elective"]
env = DuckEnv(skill, hw, num_envs=2048, device="cuda", seed=1)
env.reset()
env.randomize_episode_starts()                           # 回合进度打散，和正式训练一样
net = ActorCritic(env.obs_dim, env.critic_obs_dim, env.action_dim).to("cuda")   # 61 个观测进，14 个关节的动作出

In [ ]:
class PPOData(RolloutData):                              # data 一半
    gamma, lam = 0.99, 0.95
    num_learning_epochs, num_mini_batches = 5, 4

    def prepare(self, rollout):
        # 超时只是截断，先补上折扣后的估值，再计算优势。
        rewards = rollout["rewards"] + self.gamma*rollout["values"].squeeze(-1)*rollout["time_outs"]
        advantages, returns = compute_gae(
            rewards, rollout["dones"], rollout["values"], rollout["next_value"],
            self.gamma, self.lam)
        # 留下旧策略的概率、分布和价值估计，更新时用来比较。
        return {**flat(rollout, "obs", "critic_obs", "actions", "log_probs", "values",
                       "action_means", "action_stds"),
                "advantages": normalize(advantages).flatten(), "returns": returns.flatten(0, 1)}

    # 同一轮的样本分成四个小批，固定分组，重复用五遍。
    def batches(self, samples):
        size = len(samples["actions"]) // self.num_mini_batches
        order = torch.randperm(self.num_mini_batches * size, device=samples["obs"].device)   # 只打乱一次
        picks = [order[i * size:(i + 1) * size] for i in range(self.num_mini_batches)]
        return [{key: value[p] for key, value in samples.items()}
                for _ in range(self.num_learning_epochs) for p in picks]

In [ ]:
class PPO(PolicyModule):                                 # model 一半
    learning_rate, clip_param, desired_kl = 1.0e-3, 0.2, 0.01
    value_loss_coef, entropy_coef = 1.0, 0.01

    # 保存这批的策略变化量和当前步长，更新前据此调学习率。
    def __init__(self, net):
        super().__init__(net)
        self.kl, self.lr = torch.zeros(()), self.learning_rate

    def loss(self, batch):
        # 用当前网络评估采到的动作和状态。
        dist = self.net.action_distribution(batch["obs"])
        log_probs = dist.log_prob(batch["actions"]).sum(-1)
        entropy = dist.entropy().sum(-1).mean()
        values = self.net.value(batch["critic_obs"])
        # 用新旧策略的 KL 衡量更新幅度，不把它算进梯度。
        with torch.no_grad():
            old = torch.distributions.Normal(batch["action_means"], batch["action_stds"])
            self.kl = torch.distributions.kl_divergence(old, dist).sum(-1).mean()
        # 比较同一个动作在新旧策略下的概率，再限制比值。
        ratio = torch.exp(log_probs - batch["log_probs"])
        clipped = torch.clamp(ratio, 1 - self.clip_param, 1 + self.clip_param)
        # 在裁剪前后的策略损失中取更保守的一个。
        surrogate = torch.max(-batch["advantages"] * ratio, -batch["advantages"] * clipped).mean()
        # 价值损失也比较裁剪前后，取误差较大的一项。
        value_clipped = batch["values"] + (values - batch["values"]).clamp(
            -self.clip_param, self.clip_param)
        value_loss = torch.max((values - batch["returns"]).pow(2),
                               (value_clipped - batch["returns"]).pow(2)).mean()
        return surrogate + self.value_loss_coef * value_loss - self.entropy_coef * entropy

    # KL 偏大就减小步长，偏小就适当加大一步。
    def on_before_optimizer_step(self, optimizer):
        kl = float(self.kl)
        if kl > self.desired_kl * 2.0:
            self.lr = max(1.0e-5, self.lr / 1.5)
        elif 0.0 < kl < self.desired_kl / 2.0:
            self.lr = min(1.0e-2, self.lr * 1.5)
        for group in optimizer.param_groups:
            group["lr"] = self.lr

In [ ]:
# 先记下未训练时的摔倒次数和平均每步奖励。
before = evaluate(net, skill, hw,
                  common_step_counter=env.unwrapped.common_step_counter)   # 512 只鸭子各走 20 秒
before["falls"], round(before["mean_step_reward"], 3)

In [ ]:
# 每轮重建一次数据，也就是每轮用最新的网络重新采样
trainer = L.Trainer(max_epochs=1500, reload_dataloaders_every_n_epochs=1,
                    accelerator="gpu", devices=1, logger=False,
                    enable_checkpointing=False, enable_model_summary=False)
trainer.fit(PPO(net), PPOData(env, net))

In [ ]:
# 训练后用同样的统计口径再评一次。
after = evaluate(net, skill, hw,
                 common_step_counter=env.unwrapped.common_step_counter)
after["falls"], round(after["mean_step_reward"], 3)

In [ ]:
def load(path):                                          # 读回前两讲存下的权重
    old = ActorCritic(env.obs_dim, env.critic_obs_dim, env.action_dim)
    old.load_state_dict(torch.load(path))
    return old


# 前两讲的策略和新训的 PPO 各保留一份，按同一口径比较。
versions = {"v1 REINFORCE": load("../S03-policy-gradient/v1_reinforce.pt"),
            "v2 A2C": load("../S04-actor-critic/v2_a2c.pt"),
            "v3 PPO": net}
for name, policy in versions.items():                    # 三版放在同一个评测上
    result = evaluate(policy, skill, hw,
                      common_step_counter=env.unwrapped.common_step_counter)
    print(name, result["falls"], round(result["mean_step_reward"], 3))

In [ ]:
# 固定前进指令，录下这一版策略的表现。
record_video(net, skill, hw, "v3_ppo.mp4",
             common_step_counter=env.unwrapped.common_step_counter)
Video("v3_ppo.mp4", embed=True)

In [ ]:
# 留下 v3 权重，之后可直接载入评测和对照。
torch.save(net.state_dict(), "v3_ppo.pt")